# UEL_SGS_LucViHoangDe_C1

**Chosen approved backbone:** `FacebookAI/xlm-roberta-base`

**Team:** Lục Vị Hoàng Đế — UEL / SGS. Primary configuration: `configs/xlm_roberta_base.json`.

Fine-tune one approved encoder, evaluate macro F1 on a group-aware split, then retrain on all training rows. Attach the competition and the [approved Kaggle model](https://www.kaggle.com/models/aisuko/xlm-roberta-base/transformers/default) in Input and enable a GPU. Source code is embedded; competition IDs, labels, prompts, predictions and split data are not embedded. No GitHub checkout or external pretrained backbone is fetched. This notebook creates a CSV and never submits to Kaggle.

Prepare compatible dependencies before an offline run: PyTorch >=2.8,<3, Transformers >=4.56,<5, NumPy >=2.1,<3, pandas >=2.2,<3, scikit-learn >=1.6,<2, safetensors >=0.5. If packages are missing, install compatible versions from PyPI while internet is enabled and restart the kernel. Keep a compatible CUDA PyTorch build.

Competition Overview, Data, Challenge and Foundational Rules were verified through the official Kaggle API on 2026-10-22. Deadline: 18:00 Asia/Saigon on 2026-10-22. Public code must also be shared on the competition Kaggle forum/notebooks under an OSI-approved license; check repository publication/license status before submission. Organizer replies require all training inside Kaggle, exactly one approved model, no combining predictions from different checkpoints, and no external or synthetic training data. Only original approved pretrained weights initialize this notebook. Earlier local experiments are ineligible archives and are never loaded here. See Discussion topics 751573, 751570 and 751566. No external data, translation model or pseudo-labeling is used.

In [ ]:
import os, sys, json, base64, zlib
from pathlib import Path
from packaging.version import Version
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
MODEL_ID = 'FacebookAI/xlm-roberta-base'
CONFIG_PATHS = ['configs/xlm_roberta_base.json', 'configs/xlm_roberta_weighted.json']
MODEL_DIR = None  # Set explicitly if automatic discovery is ambiguous.
DATA_DIR = None
assert os.environ.get('KAGGLE_KERNEL_RUN_TYPE') and Path('/kaggle/working').is_dir(), 'Organizer rule: all competition training must execute inside Kaggle.'
WORK = Path('/kaggle/working')
os.chdir(WORK)
import torch, transformers, numpy, pandas, sklearn
for name, current, lo, hi in [('torch',torch.__version__,'2.8','3'), ('transformers',transformers.__version__,'4.56','5'), ('numpy',numpy.__version__,'2.1','3'), ('pandas',pandas.__version__,'2.2','3'), ('sklearn',sklearn.__version__,'1.6','2')]:
    assert Version(lo) <= Version(current) < Version(hi), f'Prepare compatible {name}; found {current}'
assert torch.cuda.is_available(), 'Enable GPU for a full training run.'
print({'torch':torch.__version__, 'transformers':transformers.__version__, 'GPU':torch.cuda.get_device_name(0)})

In [ ]:
# Source matches protector/ and configs/ in the repository.
payload = 'eNrtfYty20aW6K8gSs1e0AYhUi/byjB1HVt2PHEcl+VkaotiYUCiKcEiAQYAZWu0+vc9j37iQVLOZHeq7p3dWAT6dbr79Onzxt3eqsgrMavyYj+K0iytoihc3e6densX9H8fBFRI1rN0uhCBF6/g6UYk/Wk8u57mmfDgebmqvNkiLst0ns7iKs0zb54X3oef33z0DgYHJyH3dJFdZFF0I4oSakSRN/Iu9gbhMBxg0V7gWZDM4kU6LagrCUy6XOVF5X0qc+hlDmN6q7i6glqeLHkPjziCfMzWy9WtF5dettLvVnGWwBv4/1UiOymvFyIusnApqiKdlaqz+TAqZ3kh6rXyRCyiUiwASpylrF0VcZpFlSirqFwt0upCgRjO8qys4qzSHb99/sPZ24M3L1WFJK5iVfa5SCsRWTMMi3VWpUuhKhTi93VaiOg6vrxciEiW4mgXWSLmnlo14YsvK1FAWVb1Ti8yD/7HW/BxDTsGgF/Fi7lXXQnvBpokvGUECqwQrE8pyhKrYY0c/imoQYjjYF8fr1IoXmGreAFN4sVtCW+W6xLWPa+8qfBg4vN1KRLvc1pdebGXic9eCkCuBPyTwZLBYknIZldidr3KU3hrVjZeFCJObj3qA8H4fJUvHHBLUYX21Ph3+xL5PS41ywLIhwhjLxRXmRcxLPgIMCRECKJZeWNV8vZhNANEtCpEkhLEZQg1L/ZkL4v8MoVNH3F34/H8Yo9eRXeLeCoW9xd7dELoAdZFo8VkElZ5RLirYL5VvYRUO1zGK19V7zVqV7DBAS8ptKsjpp+twriIs0vhL0TmU7e9XuBxjfSfYjQIjwMPaiT5MgLMrcRoePgk4L43/6+s8LzOb0cK2OxyHV8K7zGsWATzfWzPQkKLSMsnz0+TL4E3TeNSISzvZrUugJgs8rjy1Zn0b8dQeRJ4Pq8yPUH31Bimd7mMv/hDmBWNVI7Gg8AbBt4BtIiB+ABMo4u9ZTwr8gsgOv8URR4l6U2KRGk06EnI8DUsIKwX/ir9Q/l+iiuFIwX8cz6EWlglUBPhHcBXCqNgoz/hJvt9Wt1BOMB/ju2JYp0V1xlisVsT4A8HdnWCBIBgCOOiiG/9MdX+FHirSc+tCfi6RpR2AKTVcuulc1n1ez23x95Q9J+eNve/dR34mfqQqxWXIiJk5AFFEinUlMDIQlquoPZSjyHhZOoA9GBkihBop+fvR9aoAD+syuDAAzwQNIiiEuV6gTTg7mJP9UrdXcBVo17A2VqkZeUDXBd7MzgSeOiFrqZBMPXqy3SxB2udZpcRYRusUYSgLdJMYAdNfAlaWuiB9ZjzYctATLORRLUPZu9Ee213IGtJ20abzcQKloiq5vmC6Ep266ul69lTKfLPtGBIcXCyvRoEdjkN2LaQMzi6cYXVLvae67Z8j30GNLgSi8TL13Bz48VJmMHDAxrBW6wCtPomzdflQt4qcNrarp7QewF3V1os8dZFXgcHotswhkpwryUe0dL9UkAnUwFHVwBCLW5hMKxVLuPFAnpGKluG3rvcg8ujr8lhdQXIB3dZ4s3TqqI2BdFfSa4INmCY7nkNDE9Qv4VsFol4BiBmjNfyuKwKmJePRWECV0Tpc2lA93BWjQ5gI+aLdXk1+lisRU+dDKK4XLXBlS1SlxsDYruKi1JoDms7e9Zke+J1AhcjvgnU/S6icj1dpiXS5MDhixSrs4SLzVckkWAo4DgreMLnxeUaV+o9lfiJKGdFSgwL0H7NlxpuVmQzYO2ABqcreWQUzVlPkRmgbsI4SRAwfiqx1wq6m+XLJZwdXn/iPhJ7RWl60Ac0pA64tQ9HAAv0QPREFWIJO1Tp93FZNvbcbAKnYLWusBEsVAzbCDDGBVzN8awq991RiUFoA44K3GoPA66jCfPHm2DbpyoON9XRF/KY6eUDASAWvo/I+cCGZlnbGs1TZIVbFpIKgLPSk6EXD1vLjiaGIDyw4fY16Gi4GbWo0bZ58vWpSRSxz23rJovqVR+2cJ2NDOF/cNPNeNDZzJn3rB1XtPxmV/yafVdktG0QVabHUC8etrTdrQzpfnhbtbiKwhWXpSG/9Afrl0raAeYPH0NJgr0RqhQkhbOYVnPD+FSd7xr6yQMq8XDR2qGkhTa7TpcYvccb3FEDWLuj69dLGmAQzQvU2EjW5ANrHPCoPgBgQ3O2wqyrmnryTQNGA/5m2OSDxbh3gKmP+FYoFZ0ALg4keiV1W3KiedkA2xz1r4PSOpQNOC0eTIFqquu6Ri1TW0c9cCkcoVcKJy3MEPdgM0d6upYwp+bBM61JbxZH6SyHwzvuxj/2mB+D4aIoA9kedXq4ZFGE7FkU6RVjbq3BTirlmGQqSatxDhMfX+z9cPbuzet3SEAu9v72/M3bHz6cPf+JH99/+OXn9x+jN+/+dvbi45tfoNLkIlMaEZTpiJE+9VKSqtPAaFlEBpSGNoKH6t1jw3evf33++kyOu6p4kHnBf1P5LDIa5iL71vslAwmiuhIl6j4FoSoKAlIfWsLSC2T5l8De43uqC2R4BUccnonv/OVGFDep+BxeZM/fw3x+OyPAlTorAXEyXUxFUe2bn30U4fpL2IEUOFQQJhb9GQppLBPFabm+zneovg8AZyWsDKxEuS+vcZijGvvT1bo/W5Sr/eXyh7MPH/skzjhjLJfUPxU8tDcEqq0zfL+tr1fxTICwef38zf6XxbJf5NgybnZZL/zKfhdwOro7ptJtPV/m+eVC7BeCJun0Jd9t60EeljiFMh745pB3ltXu/TT7xGJr/+bAxYTt9bcOHldFnq3LflnBBZrvf84X8z5UQxVu0ejQGX1z1W0Dp9Nl/xKqAKHal3/7h+Fh/2AKfZRVsZ65i9lRZ9swQEOhxgIW92f+1X/yQ/+NbNy/GYSHLqqqSmYEqtQ9zD0SjLN3L355efYB6UspKl+d957XNwd+pwk/CG4e3aW3eEu48vtVXKKMXhPftbVB/1xnKcrHxA/+GcaXdrPKOWuUgWq+LvL16qdX+SLZZlo5DzxN0Y22wLrzmF3Jp5+M8qC6UtYA/N0zr5HjhJsuXF4naeHzQ0mMbOCJL7D2UX7tMP/YhseqxBfn7oQBoU1WrgvgYMtZmo5exXDtm8sUbvLFIv8M92jGRT2sD6sOxBukrHU17z8lvljNqbyKD45PGGQ5FZyH3NJQFkvIyApjphjmK5EBw11MoUvcknlNFz1d5LNrvDJhLoW/iJfTJD715mQS8YeDgyPvkYd/AMgpGl/qyumrcL1C1sWnjlzd0lV4Jb4k6aVAnamZD1lbiEeHxSaEvQ3IYEPKWTRO8NKroXQttXn6hW3JQfHh7t5Yf6T++M6Aa/h7vP/ThO973D/+pTR36gl4CWAFAqcDmMoO7d1GZbxcLWzertZDbaR7Y0xAVivwZvlivcxK4mvkxELYrWXpO4aFFnOWWbl9b36xd4f93bPCBQT76nYlRkBLAu9aiFUk6RngpURLh81kk44ERSvCvW9GCrwaXsBKA9v0G1oIzooiL3w9/qnZHjWzO/njPoBpr4Efv2sf7l7Liw5UaQJnD5iuGeBh4vdIQd1Dbasuhlnif+kKSsXvPiEyV3sA2FPY4GvsVg/mvXnZDlEMPOCtPE1eeeqVrRBIEB4MiIA79hbENwFUsja8jYWIMIwVKOKguRavJddo1/P+yreVpqUPAWSdXWf558wzYzahIdzeDAqaCQ0cxK9/FRA0VK92JMox1pso26rS/KEpNPD4aEKZr+pO+ODZh82oIFwiRXMh82pVoBESfwj5gzvu0QHxh0eDwSDwTvS/juWzPrWLvTN1PoZHAdSWulsc7YSfgQrtS9DZpmJAxDUEVEuTnvcf/IAHYMuAH3GAfbJNgGCVg8yyiFeNXqkjnBA+8fA7dG3B+uZlCcR8PheF7lveFO5+mKsiA16LdBQR7Ep0icyBjwRXDfqt9ysaeHIU1ObOyaS6cKN+5xHPAfsJsm+J4+FagqCWFnCU00vS5mKXYc0sgrK3D4BeXJSPmU57+MdikEINHczz3aufXmA5QReiADYHJsaHMy5PvX2fA1x4IdCimsl8zosEbzEYew7MAnAJPP5nGr9zKdyVvCM4w085yOHU4Tg9Tb3H3uGkx1IyYjY7BqD1fMDoSjWRUQVR/95AqpeTxyv5lATGtDUahM8GruMJcXAG2sR0wgcGkIs4Pe/nNPsRWJj+2/MfPW2S9JZxNQOJW3ugPEfDzZd0iWWmFkrjqbiBrYN3Hl6rquF3yF1BOTKKMIh8i6K61Eiq4dclqp3El3hWqR3x/hbPZnGRhN5bttBlAqoDcwaskVemyxRkwRQobwm87OyqxSeFGUfYQrquGu4XPTUp4oLyvPJT+/R8vkoBBu5jnE7wpKU1OmgKR+q3fjWp2fhNlXTScLRIbVAAp1FHBByeDY5sTHDGPRyRfk71LKw9NqzXKk6LMlAriug8ngTwnzqvwxNvCttYel+8I9g4kBni77x36+X7W+TuF2SDZTVbJgRUy3LYIyCA3k/k5wP9z67hplGnFbaQPCPYkSX8QH/O0Z3FPzpQumUEPbukOqjoQheRR48ODwDd4Sc5w5wcKX4I+lpDpZMj5QZSazx4UOP17FpUNmcqFVV4cF09Fd+FNjlQS4wT7Dr7zm1LlaFTsy817Plkd5WM8WcNaRgTAMBPvTriwbYiWwNyvk8VyFUFyaLpMaLThIyNaVxTebp4wyAAVKmlQc+vQXpCGbpOJk0dhV0KIG5jVfjW+00A1ws0CuXRqwLIccmkAukBqurQbD8T39lIzLQA3ieIuHBFLENnfemC55G8v3p1XxmQUqs0W1vaYBTQ6GwYzx1AjhCF2mh6C/TQVyIccJbX4mDqlyGbp30jBgJqkQDFTltHvVDJUyQ5pFW1QK6L8IokhJLonoJz0oWZpKul04fMjx+PTwPvXZ4Bn/RIwf3Ym+q3Pe8vnu4CTtazo2cnTw6HQ7jeoBc/BhF5NLS6vha3dPR9POuBHGmMD9D/0Sm9RpejHj6iJxyvR48nQqX6mhqe9Ca2il3eAaVUs/RceRZGxqYIQH2DdEstsvLxDC+hG2gQeH6vV+vuk7WmpoM6e0rHBhEVPfEYf/9LI+n4U+2QWTfJyEIq5NdMkx6IbFimu266cFn9fD8yd3KLD1fnse462qZrRDN1Ej/x9ege8M2LrpYYJiDlS17p8aSnhkxlZ0TWnNMir8g632LdqJOeuXUiSwJ+CezZK3z05X0kJckRiN3AMUcxc3T4c8o/zYz5eSmqqxymOjFMMI5rBnKvSvWWxHrsnm9o9TqkQcmjU8mkve7m09bm0+3N3UZJkYNUr7kvX8NG5o5Ezbw+U8kh0HYEVn+GLbSsrJYGR9orT10JKyKGtqHwkWNxG6XXcSye/LCzRo4ZVIYXSUODdQUZTVrai7wsbWQByg0V/XFVjBt6nckEp9H2HgrSS7g9RISKvS82LBGp+KJOUCwIHJhUA9147OufjEF40KXAifJdrXgK15IqnSiM1VV67Bupny1+DZ2LUV8ktwOdzjLgcS3kidhXRqqOeMa2lsiawK6dkbdyd49kKEf3UbgapU8z4pTaZ/TJG2kWjz2JmgNbHkatYwCjWNaVhMpL8c7SF2oh36gApbh/76j65ikyLKSQpS5Q9XCqFLg13SUAiMU9rekjr2A5qIKZxzNPDV2idJ9ywWAHQ4SgKqQzOfnqRrN8DcfIJ29ysqH3goZ+NFLaHNMBP+/YBzIrbhfigV0QxMAiRUavdEoHFZGsiqe+BVSgp9jZm8guq6sIGL4C+DtR8Oa6C0asn4WWpNWPtCKEd1uqVogXNcoVlsGd3iyO2Oqyuzci7C08fq82THe92vjOcZMw8JmXgzNB2NTIOtDaZde8cpsSELypWBfFJDVDPq51AM0w6vJB69s7w4qjNmVfKVH2QTCGFQTJD/j1RFRocc/Q/D3zTo76xFuul0qd4KE6wbelTDSvEGePSo5+VaSXQHiVqI8ENXwGcsz3o6ea/fqO3UWl1oGlfaBDt46iIdR2OqnlR1IjMZucy4n0rNBpjjx3t2B+Q+vPFxSSBOOkicgO0/EVFcK7qWczIozrOhyFsQZVz7gftgCFhG/cejYmWl96Bw3933unMipD9h3+vo5B0lkIKGLa9TtBCWuI8QvhM/yPfsCvYa/p3WyRZ+QJInZuVY7NBJmKrUkz1iBc7H3rvaSgJfb4Ckjrgv++oPUkxRKJLeyUMcuXMATgERkQFzl64eE7wXvhIZmWHtsc08byeOmt1tNFitJPaA/yj3/8Q4FnGfoI0s2mvp5sbXf2mxXYRFfcGmXTNgso+phrb/KL9WCQPGEld6C1m3QcSFWMx9dStYXeCzyt7IUrL01aGtiwaZoAeKFnaX2ptaUr0wq0QOrAODorh5nNhePFHpK3DO5Wx8YuUV6wbaWo0syUipK2uMsA2uqbJJd9B9d2qmk7kmtGQqox67zEqeIKCsQ1E4y1Kb7J4j8q8kxv20k/44HL0TGqxtbz+UJIXtaJwEJAbN/kiGKlgHLgDxSNcP3UWKEzFwZazannqu755Vj3OVEXiyyQI0yaunyMwCgQWZU+n9W7CxFf2/Ye7ckmo+CYXZ5MgLc2EWtcaSzvC5ZxGIBauSaaXEeTP7taCKdawR14LW1M7J5p6EQ8KLwA4pa4eGFb2hUbaptV2bOAFSt3PNVT3IB7a9WJ+ncaJsmCIqtok4oW7TZbVM4JoiQXJRnSWKPFJqI3L40taCmKS9I+cbf06EvvUwqOYIFGuR7CizyDE5pHeUYWcFR6OMyyxCXu11xsBP6ddUW5S3+/y2Sk63BpWbrsPmohDAzBuA6I48I7gd0CaFk281ESrrklb+vHHn9zZ9rohmw782PydEEf8sXmRXDOFCmc4hshg4+aljIYhrbNYHCb+6jEUS1tnGrR1LF7dovmMh6mA+23ug3gqqynrQ4DLa4PW5BET0t7DFAE8lQwZ7e49dIkcO2/UvoFGGhMtLrSwQOYNp9KqqDP5C7WUws8jchohDLGWrJ8KlDzbCZsMMkaDqOynJaWcCtKI3iIJsAtg7/JaPcRfHYLMDjgNVZEzt7eDG3H1W93niyw03DjsUG3ZFarSzxtBmMS1Eg3+Ro0YrfaNYrF5N0m8tqCTBqDbLZbr2Qnw00aTSWi285VWCLDyc0Mygh5H5RGq6sUxJ8iR99GbDq43yGSLknjyyxHYaV0ki2wOatPlmsr1r0q1plMrGC1JDYbUzDowFjj665TLtAG4FGmBA2iNLkEVgsQnbR73uVs5yA+Fe4gcw+0+vepOjkISbIv201Sdfh8XeU/oyX+VV6cwykBhk+8cDJJBFTnI+rCQQYs2kMIzWUdGNoVyF0MWhIr1NI92Lkedku6ELDJ+DayBIiIvAlUN/UYBg7riMjOg5JG4J2RVSd5T8IGMa9wZCIOag+YMcWJiYJ50QhZ7tsKpVFD5NHavUYiL+Uw347SkKsgb3dHESu+iNma77UtORO65ulbQQf2oK7y1gp2CFrUuTAW1CQ+Hqda+vUoV6gghcAeXzkkLLTIBYacwW2fVzH6k2KwLhx/oqVoJ1gjdcI/ZNtB/Y5cBLx27u578jUcjArxFC1o34z06XfiP1po4K+oRiWBciHwMErTtB2t1WtkvBh59bBeKzJN1ibE8UYNDPKtfvaNs3heRFxTLdzGzBiyKrvgagNCDd9ULXwdSDjCbL2M0LwE22azYgFyEzX9vsGUKLDp2shms2VrelAdqmMPNR0ywOZSE37h21E+JNnTCpURUlIpUwERLauoEMscFVkwX4czYX+g0Q4U6V8/diJu0pkgKyF6kPAjxgOukxi95OayAJ+BFYjimziF2wwwQGrvoepqbXh8BB9uNp87+jrLCV68GOZZwVB1fw1KXUL0JrCvJtJKEcG92LuiswptyVhtv+UXx8ODejWbxQDBtyCzaYrqKosa0XIBOrFV1aWgvsGswKBOoK46XwXWLeMvklyOmrMYmZ+2PVMndLGJtE8r7ZBqXwLnAPAU7fX2XrAkRlcQnxVObGLDzyNY9WFDmEcy4EvGiGaAMo4CnDX06omseTLLg8k7gXo+83ZidQAnX3FdcjKhLiEuSdHVU4BpjX4WZpP6tHGBd6A0fBJyZVtXsVmo2AAA5sM+X717E5MPh7V9iE0GUmJeIlY/oeQlHZ3xaCBrjIEAsMHYDqWbWdUjadEegAQoKL93l1SbuOG3jWKM96rw7tEjKJdmUQTJWke7w3Z9JrOApVZm3uHNYy4FUhJY9x6KlfniBlaTWE/rkjo113aghFrYLRchFEHXL13Z20m3YV4z/0vK6gYXzBQZKzASK5AlSjNLniXEGzDa8OoFtbwdb/DoID1l/tbmZFFuAEZWXpnWvYbbSPFzqPa+qSlJQ+8VO2pKVms/VSNYHp3Gn5NXZs3xmt95yLijdhe5A5hJQu0QCyVlR94tRgg4AxYvJmbmsI3stlMBrmyvzdy4Xi7j4rbb4NgebQkI2aLFxFuKyI40cM9Q/kH234qJ5o0JUSSXMax0kZza7lT6PiH5MJrFsOi+q1SQ21iXXSTeu5FEfyzdmps8Tp7zQCYRwzfLGOp/Cay8bDuEAXFEaGtAENNbqdhTHDfGAo/QsUndRA9LvYUByOg2hZ4qJfuqSCLuPaa3KpOUylmEyIfD8R1uNaOg5J7OZeX4UVnSsmVmsul7PVtWQJB9bTosx5rlkuHT9n3bNGKFYZMVBQuXI7VTfFRIDpcMSR0KF4gaViiiZ7/rhqHXlq6p2WlkjOUSTLt2zQhdMxtj6UbrsmXV02YESgIn8dQ6qMu4vG7YHDBURIdqNPBQEgFs2R5DohTqziwmY/V7wmxGpoy21BHQMHkfbUK1Mdad8NKr31+HdnKJNlsuG1M3dlJaN2kmHfQPnj3DUVRvf/UOBwO2mMKP/jMuVaZMNP5iOXrSmBZDDNDgNvizP2xpRXXcZgemGf58bIMBLbj49Cs2bqOJ9t9k9xwVGPdraS6+ILGwMyjW6TFrlhtUOQoUqe2k4hvtT91BfR2GKQMiW5HGTBHQ3177IGKdiePM3Z11oJHroSU95ES6PLoXCXreKrd9DZ6DA7ueFGkuk8wKr3STMdmow5QaBEd/+b4QN6jJiGczZKiqmoVd81VA8kuoIPmzy5gEpf9TKgawWC+E1mDKez0vd9BJatVNN3+gArDb9IiS9aWo7Y8f/jP6+ZeXZ28jyiqxJf/ARfbil5/fn318gxkpovMfn0M/dkoHy30LWeHpydPBs1jEJ88Okul0evTsydP50fE8PhkcTqeDwVAMB2I6n87n4hn9eXZyJI6OxMHT45OpePLs2IoGN55g2PHw8Mmzg+PjkwNo+PRocBifzJ8cHB8dDo+ePps+OZw+nR+cPEkOjo4ODp88PUqGIh7MB/GTZHYUHx0fz06sjju09gT+wfH8ZPj06dHJcD4dxsnx8XEcH8/E8fzJXDwR0/jw+OTpwRPxdJocHw1nJ4fTZ/NjmMkJ/Hs0f3JoouNV7G+rbkoppZzj/6330/PXr9+eRT+dfXgHu/Ph13fRx/98f4bcFKKTUt8Cy32tXBSmtxKzAMUwKPD85U+hY2nxFeKE5W0JsiWIlsQ4v02zNTAYJIbk6Id/kxa5Uh+2w+HEGlKgCbQlzefF3j5PcJ8VxXs9VOagOqbXVulzXlyjrteq1rTCfODZKh3kL+osIWtWmHTM2gL1/O3b1hNpiU5AsnixQu8tarP2gZKIAgWo1+9/NUcYIV6vUOuBT3OQRPuY3TGRjjWfRXp5VbHbCK4wRbeLJESIvV9hvc5fn0dv17Pf0h9zIMYvRfRiGKar22yKwqAEwLZSKWRQIhtxzgiELvhm5LnHdpO+FkgupTEGIlXcopeQsaIYk5zQaV2Ae3P7vq+bYO9sxfGppTUmv+oy0kpkbdsC9MwAckxehKIaFhkcG3fj18Q6Od3a+bqNd7OWf0M0vdFHOEdGV0DsLARMPb1BTwG/A9d727XneHzjqkJRNHGwlPoIGI8wAlfio1RPmZRzmv3TseSAzE3C3BInn+qECnZ0PHvifjPS/e0QgPzCgpvuFpnwHNMdVXg3cmrQRPoB33/nYV5STI9dqWyi6ow5+R7kThu9c32jA30Stmx5TdevkmCqna9t3T5VRw8SWBB1eY7dwzAhPcdQWSwkhnTiUR1jFAS9LQdW4YjW6GjQVRigycfE+vy48u5UJVhpJk0WrWICVvOBo2V3eR21I67So2EzlQ9LmyVBvkWZEsmrq2ZA7TLQxiUynYEpCji2v/wXZWBps9CSVmhdoebTZotQxfVWWiJfyjO32ab7Qqrav8K+29kvjvwiXwDa5MXf0+rqfZwklOAXNQroMRgXIMkA6QDmMcKEJ9HnuFiuVxt4QZWbx9UWIZP+FVbmrblrW03PTTEo2GiOdg9e0ELdggebrIMu4oJLZ7E9VrgW0Llff3j7/Dz6+y8ffjp///zFWfTil3ev3rzm++70aPDs5NQkrPm/GosvMvrDbqaMJqeW2QrIEimXvfpdrhLJKa02SbdQi/h1Nj4qg4fqwDEvSUP1Kp9dlartocpSXs2uKJJSFQxln2hdj5rFhweScs5m6+V6wcovYBtXuucDpb6IC86BDXghBW4sFf1j6QBNXFKUiFl8a8oH4WAoywmDI1JZO+UnTvsVsFaFU27MuAqkowMVEgUcG+bdNlp2gJDzeEM1Ev90OqMUD2xtUpbdV/ctTYwrwDM8xRiEIOAal73+FzNrI/pjh5hHAHlZya+OYPbwef1OhlfaNq3s+OrUbruMJYencjpTDj44QiUnhiBmQ90fKv+zzuJXz1lCgFi2TgnKnbJp1oyZ99thszpTbmvYAXoo2FZRVzWUZrRKIeNxwGAZBJUvamgr3zaxFQOWhzusomL5Ob+Jgha2LsUbvA4kLs3whDO34O7pI4uvjocH7FMHJTbyAiSD7ZAotzarT+jM7sdhmRoOE7YLb8Le3JbviZUfaxdXE2AgTEIp9AokNwdM6zYPKXAL2dE55bihe9u3SJ7SkKVzq5tt3I/MaKMlMO4W+EjTRUMqscb0Hz2SjhjaClLz8mD3Dnm2R0eajeQsCljqW77oJr9CvYT5iGWcrckNp62QTE9WjQidGhu10L3WIjd+zd2E6uCRFVlSYpdZFk5hw65Q9ndpmWRtShE5oUIw7CVwk9XVErcHxTKguJnx3JCLxVdWzdtAckIan4ieWaRMBus4ngCMkY6nA54EkkfIyaJWoA3vSNXrZejlJGPF7ahRDrBOLslDomENBvzU8JALTwmYk8IucLg5unoDX0WhYG0Jv2THO9ENa9BZnCFZoO8eMGcuh5WB8Q4NKeLPKmIeobSSYFg+npSL3wFdhdlYvhzyDWD4NC+Vzw2qgZG5gRul1GEqluIWXrtqW4CnroqXfsQppsf5Xi5KS4R9bQ8fA2fRrJQqQmXfL6Ma+3La/gmehZjjLvtyWyh3wv6+uqkbI1EiIfh3fIoN8aM5+NBXzfvUX+900mzekrCjpVfup9b8WxTL0OGIQhN591U6Dc5ET/sIAoE3fvH2fLI/Pj97P9n/a/m9RI6wtvZwWGz8CBvd4760YogMriHSGMUViuNs8iuvjaPWRuRpoEEzmVnLMiHIYxOAi1p6JXCMZXwHBq6kEzY01zNDaBKhkyPAQ89lpAAdW1goeQtw3KXqpdbyEmUBsdRUi8wALZ2YDsZUZVILotrgAOXwIjLQSS5vLcrqUmR46PJCe8O9Vm/8XteVIgE0YqoBwow8agNC/g2IqUWFK8ibo0Hn965WKSCLWObF7ajTIy8wcxjpX509zkigBdzNRh3yrW+t4wrpQR5RVmfUzefz0VONz4DNZQ7gcxLrnnXPt/msKc9i6cRjy2DEP/ruBw1QMLUuKMonykugFcd09vz6V61o1fFs8IB1Osr510be3fWpd2N8FlHXl0WUPBR5L7qKOacIHEjsjbq13cCu2VVXHbH7WtJzA2+8rvJZDJcIj0Sq15HlMaNiWbg2iVTDEwxFxO1NRh2+Nb2WU689oZkoPXrEs63nULI+ISePt3TV5A1jEAD5Z6s1/CuteC7mAyvG6SoASriwrC7tBKuoZ5a8o/GFtHL4cyCgZFosjzH5RiWKJ5EpQ4e7wHsER+k2A1EKeSm0REkyeWqCS5xiO4mVSnNjRiVHGdcvTRWk8wZEjiGAXGgGTUHEACuNNgaeefoFFo/yJFFux++94cFT/KkyHjhpG9jIqzrbzvycq3G8cgkHuE8BP7jaHn88S0mhacZpbhfpLEWjA5mtEslzexJGlz+yffcbVgc9v74KBq9ZH2iD7lu/PLBDUEC7u7gVJ9DQT9sY1t0atquxVS27QMr4pspr7BhYJ9t35z190tS1SZNcK1XLleDIJVQSIFxpgoHOZojtiWpgQrYP4BWwrWgrUbEM5HJd+ts03mgsU5/C5CYg7nG399+hbVN+WhNDG7SmvQbezi7fZqfUjMybLiuQVYXDF5zIA7waSfBuzvMVvH6XV6/Qg1gLDPq7RFrJEEudvdbUMP9I9LwtyuHrAhf+xQ74nd7ru8cz2KemI6YAhdk5XGS7hzdEOpTE6Oi/ZuTdh6K1lh+20/pWpwIVHqSJxRK31EqTA/anGZF7t9/wqekp54AzRGw8zgd9ltrLeC6qW9KpsZ0Huk2n1BjIrYwNIBeB2DuUbbBy+IfjQpqUb2RPSmeKWqYlJ1mw05x/5eIbYbJDz2u5FhKv114tYnbH3xhRwvSZpL+aasThJJrqD7s5s4Uqo5OUIVTPLY0dbSeviHXGTSS36ZiVzuYDp51MhLU22EHbxJpcUdfsGmDY8+Pu22ZXU92qgDHOZ0IuYdM0o0d7kdv8kym/ogxzObSINn+pU/U48ralnX6+WJBKUHjKQEm0GVnVuECuyLFWG6MEgus3GLx9zz8MB94jOaVez3v0SM3d0eM6Pe2P1M9wKWKds4bdkc1wTKRZDvLl6xZ2/vBAiT0jN0JKmS90V/QmfJ7Ey7/zacZ8ADAPdLZDSW9RjBQZsy09gWPXGbnTo3eY0Z4sJKqwYTpRSmaKmymjFZSQ0p8wqYKrSKQLa3kZsXB9ZYct2n51B2FwpOwXemuOoDeErQzyTpOWVVycLQZXX08mIBdUd0Sz3ZZtC79BbddSFylQQGs74uUqfA306pxe6/t5u3Cmg03Jklrzne8IsKl7h+OXJk+lVV6yFZtjYtiyjqFYBLz15Xs3TRYyDqc2g4HLK4VTtCP4g96GoA7mP5B8teRscz69ax3DnUKCHiZ6tYyuFZ8cj8Yk3Yols1tYIzjNmFJaOlSGRpHVLXA4BAK7U1TEzhVg5A5KKGekW7snS6yRVqVGmNRGTtj1/O+IE1tbX/JlXLW+Ok5hcaiqkqIEoDF/K3xgAjM5iGhumexAwLljzLkHyaGOB/cyzPOuQUjuWcMiyn0iAghSIxQJ9TFMMUxW3sClHaSRbvAcNJIdrqAJBsXIRMiTYJ441HVlsLX2oPIgE61d5CXnacrY6B3YNhICINoYSErpkYE0BkZLZRhLezkCdxakeVdMLWuiVvnKN9qn3mZN1p+hA6spvDj5npq97+M0MTk5mQe6rwhDnZuFda3+R5CN+bO3lCQHhTSrjUzcS7zCVXzDjluzvEAZlel62AoxYAFecYAfnYAEXvPe69sz7v0bKP90wHBN+Se1em0NSsPDZCBNAMXDOccLznAZoacqhie4akEVoqF4jpFD85r2CiSXUnFbztFaKXwcuW0KUlDHcQAB36NooBUTQHH73JwBjytJCu7oz706Une4Y/cNn2i+2EP6Q+MDATQI0COjLvAHiV9rZ598NKfhXzoBPqIuogWvRn04JBGPR90VUFGJJ+Qv3ehPvLInKQY+NJCwzQDI81xn9CMynFFLxm298+yAN1ukK6KEEeb2jFr5T6D/bam75YfuIxqU4mQICuQreMV7nZDi5DZCqWbE8dy9VnNmczyMdXLB6jAmajaTIWnp/4F3hb27sGlDvYN/wXAt2tTTtiWsReNizDZfhKeM6Mg5EIoz54eXiHwhaiyXxA4KNAJ0NXHzDjLvE5IG3jEHfS/iFeYX2hRrDzTPuuN6923Rwk7igOYEFIASqjZ4Nsf7uzDU4v7Z09GRwI0U3sLB1faBK21Iu2B6aUmvYABoz7Bg2TmiRq4FuWxuigTKDYcduq9rqLbUztRtldFZAA0BwBaJ/tON38OwepKC36hNLqNPSkjlcmsqg3Ze09VHa6byKwLS5ZYqcGuzYKqFmetszZgFiPsldlTWAdIXKWb8ZTN8y3k2pv8dO+51fgK5o8GmfDpBTQj8us6V1iRii4PuWwugO3WLyMRSu5NdonHQm/k/aniJzoV8XLU8ed+kolylhV4SUu9yPrqnotIKqHnwc8sFkK1o2+221rGW3FG2upV5LNtPt8xaVY9A3UQfdGqr2oi2Adc2Dlm5bOgWHmw87lzL9RDaJnlj9BO3+17TBu20W/viRiHia0cNMm7kyeKPOTUpvKbt9fYr6DO6XK3ReS/nzDGXU6cf29fvi3SdMLV91FfhZ0QfPToMvMPtao7B14nPOmdG4FmqKc1qBJJh+fMzajC0xiTP969J/UJ+KxjqJ9NgOcYD+WBbFdycbhsShZkOviYb2MYUcAhuz3zdozX5l1NHZSqtZ/+y56ydT5QyRqY+knDZeW7kK6Jy/B3hfF0YhVy3HVMlrqGlcr/QIDU69rIF7W4Btc84qLkaEi9Jwrb0OjyIZJ54Bi1GWCaU/Gys5Yz0xuOCWrdZmE0bOLxJ/jnD9efzarlBy/a7uEA3LPduv2N7xyb8VVhn07Yb6snkZuNBLcOxyukDfJI2EVufX1UB7Du5vhBWtvm5WE40kthbB9XO/qaaYg4C6tmJXG7WVFtsSlqzJDa+S9p5uv43cgXuSh2Y0UHYXFDaOSHHy0CWKJRGGc5CWFm6GWH/WG7HFwZkjYANV5FU5y/QPiUq9uQP+if8P5IHUUV5h3Wr/zedZv+dt40N+zI5sUtD6rmRjbfV/0hixlYD+gbL8u6pCHeyMNdkZ0pMRYZmla5K742maqf2h/cW6bRQ2TzrwTS6RW/n68SMr3KfWEOMURvBUfoRVjSfJ9Pq+LbkLFJlit8uxOQwKq/WUH1BcAlXR0rfDCUs+2e64gzfRreKX+uV2akUsEyMxyZ7jGenzUZTvOz265yxZFdt2eTcfBxteVtaRCT+AJMjHDmCkZOVtS11/EYQaqTedDZ27KuThzoWOh3ZMjStNrB3VsZrw9i1tDW5LWkh2OKJndjGxZZ2EsvQzgO/tN1wkxxiVicyXWnJxEq52PEpEzsL6KbvmfwPZgK0PnugOSEy/eBXrg0rZKeBboopXWzRv0/yaM09uVFz0M3Y+ABM/kweKZ3/aempn0tPSmuNZFQWk1RkqtCgQ6o31C14+RyuerTrUY4UfYtJXc+GdW9opR6y/MpyPKKBxlqbNdldxqyLjfrOeEDaVUvxYivPJEy1zLl0FiKpljWyqYrS/cOSo+V94HorMGra7yb88eVKsDs4J0hlpk6lxiB2VUtLVs4Mvf18D+QZfbBieMRfrKDvrFDKXO+D+pC6dWtI9SHchXl2WeJXoShvKy09lTODQvZHZJXn68WiT0G4zBbpmI0dhDX+/nCHqGbRraa01ql5btAomQOEASj3KxEvCZsx/wf5DwH+Zin59ODXYXGxfz17K/NnYeLb5Wpd8vvz1+f6PfbDb+3kR7o4SUs437fk+COrXayHQhzPvN/wx2zq/ZjjV7/EABgW+DEcDvD9dK67wM3HPGUREJqF0JC1JVxCMeveneeXxTKSqc0iTG1Wn7OtgtmSE03C46YnxgQVciWcfNVOtKOsYQ7UoXxj+FZ8O1Rd1XhaanEgy1ocGRAMWer47dHnO0JA9oEqtn32ZOlQFVnua7LoxG0lA9KpSJagnIdvjtQA7e63lJ2R4uO4llLq2pBbUiL1KF83HAl1Xxu3mkEWyf/f7n/Fdh//22z3/X8D9zI7fg=='
embedded = json.loads(zlib.decompress(base64.b64decode(payload)))
assert all(p.startswith(('protector/', 'configs/')) for p in embedded)
for relative, contents in embedded.items():
    path = (WORK / relative).resolve()
    assert path.is_relative_to(WORK.resolve())
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents, encoding='utf-8')
sys.path.insert(0, str(WORK))
print('Embedded source files:', sorted(embedded))

In [ ]:
from protector.constants import APPROVED, ENCODERS
from protector.data import read_data, sha256, audit_data, validate_submission
from protector.training import train_experiment, finalize
from protector.runtime import require_kaggle_runtime, verify_competition_input, verify_pretrained_input
require_kaggle_runtime(MODEL_ID)
assert MODEL_ID in ENCODERS
assert all(json.loads((WORK/p).read_text())['model_id'] == MODEL_ID for p in CONFIG_PATHS)
input_root = Path('/kaggle/input')
if DATA_DIR is None:
    roots = [p.parent for p in input_root.rglob('train.csv') if (p.parent/'test.csv').exists() and (p.parent/'sample_submission.csv').exists()]
    assert len(roots) == 1, f'Set DATA_DIR explicitly; candidates: {roots}'
    DATA_DIR = roots[0]
DATA_DIR = Path(DATA_DIR)
verify_competition_input(DATA_DIR)
train, test, sample = read_data(DATA_DIR)
audit_data(DATA_DIR, WORK/'artifacts/audit')
if MODEL_DIR is None:
    slug = APPROVED[MODEL_ID].split('/')[1]
    candidates = [p.parent for p in input_root.rglob('config.json') if slug in str(p.parent).lower()]
    assert len(candidates) == 1, f'Attach the approved Kaggle model or set MODEL_DIR. Candidates: {candidates}'
    MODEL_DIR = candidates[0]
MODEL_DIR = Path(MODEL_DIR)
verify_pretrained_input(MODEL_DIR, MODEL_ID)
assert (MODEL_DIR/'config.json').exists()
print('Data:', DATA_DIR, 'Model:', MODEL_DIR, 'Rows:', len(train), len(test))

In [ ]:
runs = []
for config_path in CONFIG_PATHS:
    output = WORK/'artifacts'/('kaggle_' + Path(config_path).stem)
    if (output/'history.json').exists():
        meta = json.loads((output/'run.json').read_text())
        assert meta.get('execution', {}).get('platform') == 'kaggle'
        assert 'total_seconds' in meta, 'Incomplete earlier run: use a fresh output directory.'
        assert meta['provenance']['train_file_sha256'] == sha256(DATA_DIR/'train.csv')
        assert meta['provenance']['split_sha256'] == sha256(WORK/'artifacts/audit/split.csv')
        assert meta['provenance']['config_file_sha256'] == sha256(WORK/config_path)
    else:
        train_experiment(DATA_DIR, WORK/'artifacts/audit/split.csv', WORK/config_path, MODEL_DIR, output)
    metric = json.loads((output/'metrics.json').read_text())
    runs.append((metric['macro_f1'], output))
    print(output.name, metric)
selected_score, selected_run = max(runs, key=lambda item: item[0])
if len(runs) == 2 and abs(runs[0][0] - runs[1][0]) < 0.005:
    # Compare optimization stability; predictions/checkpoints are never combined.
    confirmed = []
    for config_path, (score42, original_run) in zip(CONFIG_PATHS, runs):
        config = json.loads((WORK/config_path).read_text())
        config['seed'] = 43
        confirmation_config = WORK/'configs'/(Path(config_path).stem + '_seed43.json')
        confirmation_config.write_text(json.dumps(config, indent=2))
        confirmation_run = WORK/'artifacts'/('kaggle_' + confirmation_config.stem)
        if not (confirmation_run/'run.json').exists():
            train_experiment(DATA_DIR, WORK/'artifacts/audit/split.csv', confirmation_config, MODEL_DIR, confirmation_run)
        confirmation_meta = json.loads((confirmation_run/'run.json').read_text())
        assert 'total_seconds' in confirmation_meta and confirmation_meta['execution']['platform'] == 'kaggle'
        score43 = json.loads((confirmation_run/'metrics.json').read_text())['macro_f1']
        confirmed.append(((score42 + score43)/2, original_run))
    _, selected_run = max(confirmed, key=lambda item: item[0])
    selected_score = json.loads((selected_run/'metrics.json').read_text())['macro_f1']
    (WORK/'artifacts/selection.json').write_text(json.dumps({'method':'mean validation F1 across seeds 42 and 43; no ensembling', 'scores':[(s,str(p)) for s,p in confirmed], 'selected_run':str(selected_run)}, indent=2))
else:
    (WORK/'artifacts/selection.json').write_text(json.dumps({'method':'validation F1; difference >= 0.005 or one configuration', 'selected_run':str(selected_run)}, indent=2))
print('Selected single configuration:', selected_run, 'seed-42 validation macro F1:', selected_score)

In [ ]:
from protector.diagnostics import evaluate_lengths
evaluate_lengths(selected_run, DATA_DIR, WORK/'artifacts/audit/split.csv', WORK/'artifacts/kaggle_length_ablation')

In [ ]:
final_dir = WORK/'artifacts/final'
if not (final_dir/'submission_validation.json').exists():
    finalize(DATA_DIR, selected_run, MODEL_DIR, final_dir)
else:
    final_meta = json.loads((final_dir/'run.json').read_text())
    assert final_meta.get('execution', {}).get('platform') == 'kaggle'
    assert final_meta['provenance']['selected_experiment'] == str(selected_run.resolve()), 'Existing final output used another experiment; use a fresh output directory.'
validation = validate_submission(final_dir/'submission.csv', DATA_DIR)
print(json.dumps(validation, indent=2))
from IPython.display import display, FileLink
display(FileLink(str(final_dir/'submission.csv')))
print('No Kaggle submission was made. Review the CSV before authorizing submission.')

Validation scores belong to the selection model trained on the training partition. The final model is retrained on all 14,000 rows for the selected epoch count; its hidden-test score is unknown. CSV validity does not establish hidden-test performance. Class order: BENIGN, JAILBREAK, PROMPT_INJECTION. Prompt strings are always untrusted data. CUDA attention can cause small numerical differences even with fixed seeds.